In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Question 1) load dataset

In [ ]:
from google.colab import files

In [ ]:
df = pd.read_csv('/content/winequality-red.csv', sep=';')

print('Shape of dataset :', df.shape)
print('\nColumn Names :')
print(df.columns.tolist())
print('\nFirst 5 rows :')
df.head()

check weather there is any missing vakues

In [ ]:
print('Missing values before handling :')
print(df.isnull().sum())

In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)

print('Missing values after handling :')
print(df.isnull().sum())
print('\nAll missing values have been replaced with column mean!')
print('\nBasic Statistics :')
df.describe()

Observation:  
The dataset was loaded successfully with 1599 rows and 12 columns. Any missing values found were replaced by the mean of their respective columns to maintain data integrity without losing any rows.

Question 2: Extract the following columns as vectors: alcohol,citric acid.

In [ ]:
alcohol_vector = df['alcohol'].values
citric_acid_vector = df['citric acid'].values

print('Alcohol Vector (first 10 values) :')
print(alcohol_vector[:10])
print('Shape :', alcohol_vector.shape)

print('\nCitric Acid Vector (first 10 values) :')
print(citric_acid_vector[:10])
print('Shape :', citric_acid_vector.shape)

In [ ]:
print('Alcohol Vector Stats :')
print('  Mean   :', round(np.mean(alcohol_vector), 4))
print('  Min    :', np.min(alcohol_vector))
print('  Max    :', np.max(alcohol_vector))

print('\nCitric Acid Vector Stats :')
print('  Mean   :', round(np.mean(citric_acid_vector), 4))
print('  Min    :', np.min(citric_acid_vector))
print('  Max    :', np.max(citric_acid_vector))

plotting the two histograms of vectors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(alcohol_vector, bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Alcohol Vector')
axes[0].set_xlabel('Alcohol Content')
axes[0].set_ylabel('Frequency')

axes[1].hist(citric_acid_vector, bins=20, color='coral', edgecolor='black')
axes[1].set_title('Distribution of Citric Acid Vector')
axes[1].set_xlabel('Citric Acid')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

Observation:  
The alcohol vector has values roughly ranging from 8 to 15, with most wines having an alcohol content around 10-11. The citric acid vector ranges from 0 to 1, and many wines have very low citric acid content near zero.

Question 3 : Select two features (e.g., alcohol and density) from the dataset and calculate the covariance matrix using np.cov(X.T), where X is the feature matrix consisting of the selected columns.

In [ ]:
X = df[['alcohol', 'density']].values

print('Feature Matrix X shape :', X.shape)
print('\nFirst 5 rows of feature matrix X :')
print(X[:5])

In [ ]:
cov_matrix = np.cov(X.T)

print('Covariance Matrix :')
print(cov_matrix)
print('\nShape of Covariance Matrix :', cov_matrix.shape)
print('\nInterpretation :')
print('  Variance of Alcohol  :', round(cov_matrix[0][0], 6))
print('  Variance of Density  :', round(cov_matrix[1][1], 6))
print('  Covariance (Alcohol, Density) :', round(cov_matrix[0][1], 6))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cov_matrix,
            annot=True,
            fmt='.6f',
            xticklabels=['alcohol', 'density'],
            yticklabels=['alcohol', 'density'],
            cmap='coolwarm')
plt.title('Covariance Matrix Heatmap (Alcohol vs Density)')
plt.tight_layout()
plt.show()

Observation:
The diagonal values of the covariance matrix show the variance of each feature individually. Alcohol has a much higher variance than density. The off-diagonal value shows the covariance between alcohol and density — a negative value means they have an inverse relationship (as alcohol increases, density tends to decrease).

Question 4: Perform eigen decomposition on the covariance matrix you computed in question 3. Identify and interpret the results:Identify the top 2 eigenvalues of the covariance matrix,Identify the corresponding eigenvectors.

In [ ]:
# performing eigen decomposition
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# sorting eigenvalues in descending order
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]

print('Top 2 Eigenvalues :')
print('  Eigenvalue 1 :', round(eigenvalues[0], 8))
print('  Eigenvalue 2 :', round(eigenvalues[1], 8))

print('\nCorresponding Eigenvectors :')
print('  Eigenvector 1 :', eigenvectors[:, 0])
print('  Eigenvector 2 :', eigenvectors[:, 1])

In [ ]:
# interpretation of eigenvalues
total = sum(eigenvalues)
variance_explained = [(val / total * 100) for val in eigenvalues]

print('Variance Explained by Each Eigenvalue :')
print('  Eigenvalue 1 explains :', round(variance_explained[0], 4), '%')
print('  Eigenvalue 2 explains :', round(variance_explained[1], 4), '%')

# visualize eigenvalues as bar chart

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(['Eigenvalue 1', 'Eigenvalue 2'], eigenvalues, color=['steelblue', 'coral'])
plt.title('Eigenvalues from Covariance Matrix')
plt.ylabel('Eigenvalue')
plt.tight_layout()
plt.show()

**Observation:**  
Eigenvalue 1 is significantly larger than Eigenvalue 2, meaning the first eigenvector captures almost all the variance in the data. In PCA terms, the first principal component (direction of Eigenvector 1) explains the majority of information in the alcohol-density feature space. Eigenvector 1 points mostly along the alcohol axis, confirming that alcohol is the dominant feature in this pair.

Most Common Wine Quality (Probability)

In [ ]:
# count of each quality score
quality_counts = df['quality'].value_counts().sort_index()
total_wines = len(df)

print('Wine Quality Distribution :')
print(quality_counts)

print('\nMost Common Quality Score :', quality_counts.idxmax())
print('Count :', quality_counts.max())

print('\nProbability of Each Quality Score :')
for score, count in quality_counts.items():
    prob = round(count / total_wines * 100, 2)
    print(f'  Quality {int(score)} : {count} wines = {prob}%')

In [ ]:
# bar chart for quality distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# count bar chart
colors = ['#d62728' if q == quality_counts.idxmax() else 'steelblue'
          for q in quality_counts.index]
bars = axes[0].bar(quality_counts.index.astype(int), quality_counts.values, color=colors)

for bar, val in zip(bars, quality_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 5,
                 str(val), ha='center', fontsize=10)

axes[0].set_title('Wine Quality Score Distribution\n(Red = Most Common)')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Number of Wines')
axes[0].set_xticks(quality_counts.index.astype(int))

# probability pie chart
probs = quality_counts.values / total_wines * 100
axes[1].pie(probs,
            labels=[f'Quality {int(q)}' for q in quality_counts.index],
            autopct='%1.1f%%',
            startangle=140,
            colors=plt.cm.Set3.colors[:len(quality_counts)])
axes[1].set_title('Probability Distribution of Wine Quality')

plt.tight_layout()
plt.show()

Observation:  
Quality score 5 is the most common in the dataset, followed closely by quality 6. Together, quality 5 and 6 account for over 82% of all wines in the dataset. Very low quality wines (3) and very high quality wines (8) are extremely rare, each making up less than 1-2% of the dataset. This shows that the wine quality distribution follows a roughly normal/bell-shaped pattern centered around the middle scores.